## Deliverable 2

task 1:

In [ ]:
from pathlib import Path 

import re

def read_file(file_path):

    path = Path(file_path)

    try: 
        file_text_content = path.read_text(encoding="utf-8")
        return(file_text_content, 'utf-8')

    except UnicodeDecodeError:
        file_text_content = path.read_text(encoding="iso-8859-1")
        return(file_text_content, 'iso-8859-1') 

def parse_log(log_text):
    pattern = re.compile(
        r"\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})] "
        r"\[(?P<register_id>[^\]]+)\] "
        r"item=(?P<item>\w+) "
        r"price=(?P<price>\d+\.\d+) "
        r"(?P<currency>[A-Z]{3}) "
        r"\((?P<status>SUCCESS|REFUND|CANCELLED)\) "
        )
    records = []
    for line in log_text.splitlines():
        match = pattern.search(line)

        if match:
            record = match.groupdict()

            record["price"] = float(record["price"])

            records.append(record)
    return records 

data_dir = Path("data")
all_records = []
utf8_count = 0
iso_count = 0

# Finner alle .log-filer i data-mappen og undermapper
log_files = list(data_dir.rglob("*.log"))

for file_path in log_files:
    # Les og pars hver fil
    content, encoding = read_file(file_path)
    records = parse_log(content)
    
    # Samle resultatene
    all_records.extend(records)
    
    if encoding == "utf-8":
        utf8_count += 1
    else:
        iso_count += 1
        
    print(f"Fil: {file_path.relative_to(data_dir)} | Tegnsett: {encoding:<10} | Fant: {len(records)} transaksjoner")

print("\n" + "="*50)
print(f"Totalt antall filer testet: {len(log_files)}")
print(f"Filer lest med UTF-8:        {utf8_count}")
print(f"Filer lest med ISO-8859-1:  {iso_count}")
print(f"Totalt gyldige transaksjoner hentet ut: {len(all_records)}")
print("="*50)

Brukt tegnsett: utf-8
Antall parsete transaksjoner: 0


[]

Task 2: 


In [ ]:
import json

def find_log_files(dir_path): 
    dir_path = Path(dir_path)
    log_files = []


    for item in dir_path.iterdir():
        if item.is_dir(): 
            log_files.extend(find_log_files(item))

        elif item.is_file() and item.suffix == ".log": 
            log_files.append(item)
    return log_files

def find_active_registers(data):
    active_registers = set()

    if isinstance(data, dict):
        if "id" in data and data.get("active") is True:
            active_registers.add(data["id"])

        for value in data.values():
            active_registers.update(find_active_registers(value))


    elif isinstance(data, list):

        for item in data: 
            active_registers.update(find_active_registers(item))

    return active_registers

with open("data/store_config.json", "r", encoding="utf-8") as f: # not sure if this is needed? 
    metadata = json.load(f)

active_registers_in_metadata = find_active_registers(metadata)
        

Task 3: 

In [3]:
import numpy as np


def removing_price_outliers(data): 
    
    price = np.array(data, dtype=float).ravel()
    mean = np.mean(price)
    std = np.std(price)

    outliers = np.abs(price - mean) > 3 * std

    valid_prices = price[~outliers]
    valid_median = np.median(valid_prices)

    price[outliers] = valid_median 

    return price


def compute_refund_ratio(sales_count: np.ndarray,refund_count: np.ndarray) -> np.ndarray:

    refund_ratio = refund_count/(sales_count + 10**-6) 
    
    return refund_ratio


Task 4: 

In [ ]:
import pandas


def convert_transaction_dicts_df(records):
    df = pandas.DataFrame(records)

    if "timestamp" in df.columns:
        df['timestamp'] = pandas.to_datetime(df["timestamp"])

        df = df.sort_values(by="timestamp", ascending=True, ignore_index=True)

    return df 

def total_price_per_hour(df): 

    per_hour = df.groupby(["item", pandas.Grouper(key="timestamp", freq="1h")])["price"].sum().reset_index()

    return per_hour

def create_pivot_table(df): 
    pivot_table = df.pivot_table(
        index="item",
        columns="status",
        values="price",
        aggfunc="sum",
        fill_value=0
    )
    return pivot_table
